In [22]:
from dotenv import load_dotenv

load_dotenv()

True

In [26]:
from langchain.agents import AgentState

class CustomState(AgentState):
    favourite_colour: str

## Write to state

In [28]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they've revealed it."""
    return Command(update={
        "favourite_colour": favourite_colour, 
        "messages": [ToolMessage("Successfully updated favourite colour", tool_call_id=runtime.tool_call_id)]}
        )

In [30]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    "gpt-5-nano",
    tools=[update_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [32]:
from langchain.messages import HumanMessage

response = agent.invoke(
    { "messages": [HumanMessage(content="رنگ مورد علاقه من سبز است.")]},
    {"configurable": {"thread_id": "1"}}
)

In [33]:
print(response["messages"][-1].content)

مرسی! رنگ مورد علاقه شما سبز است و ثبت شد. اگر دوست دارید می‌توانم از این رنگ برای پیشنهادها یا تم/interface استفاده کنم یا رنگ‌های دیگری هم اضافه کنم. آیا کار دیگری با رنگ‌ها دارید که بخواهید انجام دهم؟


In [38]:
response = agent.invoke(
    { 
        "messages": [HumanMessage(content="سلام. حالت چه طوره؟")],
        "favourite_colour": "red"
    },
    {"configurable": {"thread_id": "10"}}
)

print(response["messages"][-1].content)

سلام! ممنون، من خوبم. شما چطورید؟ چه کمکی از دستم برمیاد یا دوست دارید در چه زمینه‌ای صحبت کنیم؟ اگر دوست دارید فارسی تمرین کنیم یا ترجمه‌ای بخواهید بگویید.


## Read state

In [40]:
@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"

agent = create_agent(
    "gpt-5-nano",
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [42]:
response = agent.invoke(
    { "messages": [HumanMessage(content="رنگ مورد علاقه من سبز است")]},
    {"configurable": {"thread_id": "1"}}
)

print(response["messages"][-1].content)

عالی! رنگ مورد علاقه شما سبز است و ثبت شد.

دوست دارید با این رنگ چه کاری انجام بدهم؟ مثال‌ها:
- پیشنهاد پالت‌های رنگی متناسب با سبز
- ترکیب‌های رنگی همراه با سبز (مثلاً با قرمز، آبی یا طلایی)
- سفارشی‌سازی تم و پس‌زمینه برای گفتگوها با تم سبز

هر کدام را که بگویید انجام می‌دهم.


In [44]:
response = agent.invoke(
    { "messages": [HumanMessage(content="رنگ مورد علاقه من چیه؟")]},
    {"configurable": {"thread_id": "1"}}
)

print(response["messages"][-1].content)

رنگ مورد علاقه شما: سبز.

اگر می‌خواهی رنگ را تغییر بدهی یا پالت‌های سبز (یا ترکیبات با رنگ‌های دیگر) را ببینم، بگو تا انجام بدهم.


In [ ]:
response